GET RESULTS

In [1]:
from experiment_main import *

from math import floor
import csv
from os import listdir  # , remove

# sns.set()
import time

# mode = "1_day"
alpha_range = []
gamma_range = []
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)
gen = [(x, y) for x in alpha_range for y in gamma_range]

In [2]:
prefix_tex_tables = "/media/calavera/1TB HDD/Projects/MACD Project/Paper/paper_ready/table_image/"
prefix_text = f"./results/rules"

macd_params_list = [
    (12, 26, 9),
    (9, 19, 7),
    (14, 30, 9),
    (10, 22, 7),
    (6, 13, 4),
    (19, 39, 9),
    (8, 24, 9),
    (8, 17, 9),
    (5, 15, 9),
    (3, 10, 16),
    (5, 13, 3),
]
market = "stocks"
trade_cost = False
time_formats_list = ["1_day", "1_hour", "5_minute"]
experiments_list = ["one_pos", "simulation"]
signal_if_list  =[(False, False), (True, False), (False, True)]
data_files = [
    "aapl.us.txt",
    "amgn.us.txt",
    "amzn.us.txt",
    "axp.us.txt",
    "ba.us.txt",
    "cat.us.txt",
    "crm.us.txt",
    "csco.us.txt",
    "cvx.us.txt",
    "dis.us.txt",
    "fb.us.txt",
    "googl.us.txt",
    "gs.us.txt",
    "hd.us.txt",
    "hon.us.txt",
    "ibm.us.txt",
    "jnj.us.txt",
    "jpm.us.txt",
    "ko.us.txt",
    "mcd.us.txt",
    "meta.us.txt",
    "mmm.us.txt",
    "mrk.us.txt",
    "msft.us.txt",
    "nflx.us.txt",
    "nke.us.txt",
    "nvda.us.txt",
    "pg.us.txt",
    "shw.us.txt",
    "trv.us.txt",
    "unh.us.txt",
    "v.us.txt",
    "vz.us.txt",
    "wmt.us.txt",
]

data_files.remove("meta.us.txt")
data_files.remove("fb.us.txt")

sorted_ticks = sorted(data_files)
data_files

['aapl.us.txt',
 'amgn.us.txt',
 'amzn.us.txt',
 'axp.us.txt',
 'ba.us.txt',
 'cat.us.txt',
 'crm.us.txt',
 'csco.us.txt',
 'cvx.us.txt',
 'dis.us.txt',
 'googl.us.txt',
 'gs.us.txt',
 'hd.us.txt',
 'hon.us.txt',
 'ibm.us.txt',
 'jnj.us.txt',
 'jpm.us.txt',
 'ko.us.txt',
 'mcd.us.txt',
 'mmm.us.txt',
 'mrk.us.txt',
 'msft.us.txt',
 'nflx.us.txt',
 'nke.us.txt',
 'nvda.us.txt',
 'pg.us.txt',
 'shw.us.txt',
 'trv.us.txt',
 'unh.us.txt',
 'v.us.txt',
 'vz.us.txt',
 'wmt.us.txt']

In [3]:
crossover_list = ["MACDCROSS", "ZEROCROSS"]
rules_list = ['rule 1', 'rule 2', 'rule 3', 'Genetic']
overall_dict = {macd_params: {crossover_type: [] for crossover_type in crossover_list} for macd_params in macd_params_list}
for macd_params in macd_params_list:
    params_directory = "_".join(str(val) for val in macd_params)
    for time_format in time_formats_list:
        for experiment_name in experiments_list:
            for signal_line_if, signal_line_if_target in signal_if_list:
                signal_lines_ifs_joined = "_".join(
                    str(val) for val in [signal_line_if, signal_line_if_target]
                )
                prefix = f"./results/{params_directory}/{time_format}/{experiment_name}/{signal_lines_ifs_joined}/"
                aggregate_dict = {crossover_type: {rule: [0, 0, 0] for rule in rules_list} for crossover_type in crossover_list}
                for ticker in data_files: # aggregate
                    file_read = load_json_file(prefix + ticker + "_chosen_rules")
                    # print(file_read)
                    
                    # "MACD", "0", "0" | "SMACD", a, g | "cmacd", alpha, gamma
                    for crossover_type in crossover_list:
                        for rule in rules_list:
                            for i in range(3):
                                aggregate_dict[crossover_type][rule][i] += file_read[crossover_type][rule][0][i]
                for crossover_type in crossover_list:
                    aggregate_dict[crossover_type]['total'] = []
                    for rule in rules_list:
                        aggregate_dict[crossover_type]['total'].extend(aggregate_dict[crossover_type][rule])
                    overall_dict[macd_params][crossover_type].append(aggregate_dict[crossover_type]['total'])

In [4]:
aggregate_dict

{'MACDCROSS': {'rule 1': [33, 431, 7376],
  'rule 2': [17, 1253, 6570],
  'rule 3': [3, 463, 7374],
  'Genetic': [2, 306, 7532],
  'total': [33, 431, 7376, 17, 1253, 6570, 3, 463, 7374, 2, 306, 7532]},
 'ZEROCROSS': {'rule 1': [47, 177, 7616],
  'rule 2': [39, 635, 7166],
  'rule 3': [12, 145, 7683],
  'Genetic': [6, 158, 7676],
  'total': [47, 177, 7616, 39, 635, 7166, 12, 145, 7683, 6, 158, 7676]}}

In [5]:
overall_dict[(12,26,9)]['MACDCROSS']

[[12, 122, 250, 8, 230, 146, 1, 95, 288, 2, 50, 332],
 [28, 127, 229, 15, 212, 157, 11, 112, 261, 0, 50, 334],
 [28, 138, 218, 15, 215, 154, 9, 122, 253, 2, 55, 327],
 [14, 124, 246, 7, 227, 150, 0, 98, 286, 0, 43, 341],
 [27, 124, 233, 15, 208, 161, 11, 107, 266, 1, 40, 343],
 [27, 139, 218, 15, 213, 156, 9, 117, 258, 0, 50, 334],
 [61, 606, 3109, 30, 1122, 2624, 3, 335, 3438, 8, 196, 3572],
 [104, 789, 2883, 40, 1257, 2479, 4, 420, 3352, 2, 233, 3541],
 [95, 791, 2890, 36, 1205, 2535, 4, 440, 3332, 2, 220, 3554],
 [59, 617, 3100, 25, 1137, 2614, 3, 338, 3435, 3, 195, 3578],
 [108, 795, 2873, 42, 1251, 2483, 4, 421, 3351, 2, 234, 3540],
 [99, 795, 2882, 34, 1217, 2525, 4, 439, 3333, 3, 224, 3549],
 [22, 199, 7619, 15, 755, 7070, 4, 149, 7687, 1, 105, 7734],
 [60, 627, 7153, 27, 1172, 6641, 2, 378, 7460, 6, 204, 7630],
 [63, 608, 7169, 31, 1119, 6690, 1, 397, 7442, 4, 251, 7585],
 [18, 210, 7612, 14, 775, 7051, 3, 158, 7679, 1, 122, 7717],
 [62, 637, 7141, 27, 1163, 6650, 1, 378, 7461,

In [ ]:
for macd_params in macd_params_list:
    params_str = "_".join(str(val) for val in macd_params)
    for crossover_type in crossover_list:
        time_formats_list = ["1_day", "1_hour", "5_minute"]
        experiments_list = ["one_pos", "simulation"]
        signal_if_list  =[(False, False), (True, False), (False, True)]
        sys_modes = ["System 1", "System 2", "System 3"]
        rules_list = ["Rule 1", "Rule 2", "Rule 3", "GA"]
        macd_names_list = ['MACD', 'SMACD', 'CMACD']
        col_names = [x + " " + y for x in rules_list for y in macd_names_list]

        data = overall_dict[macd_params][crossover_type]
        # data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

        tuples = []
        for time_format in time_formats_list:
            for experiment_name in experiments_list:
                for sys_mode in sys_modes:
                    tuples.append((time_format, experiment_name, sys_mode))
        table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

        table_data = pd.DataFrame(
                    data,
                    columns=col_names,
                    index=table_index,
                )

        # to_csv does not create directories, so on a fresh results tree this died with
        # OSError: Cannot save file into a non-existent directory. write_rules_tex writes
        # into the same folder, so one makedirs covers both.
        os.makedirs(f"./results/rules/{params_str}", exist_ok=True)
        table_data.to_csv(f"./results/rules/{params_str}/rules_{crossover_type}.csv", index=True)
        crossover_name = "MACD Cross" if crossover_type == "MACDCROSS" else "Zero Cross"
        caption = f"Chosen rules for {macd_params} paramters {crossover_name}"
        write_rules_tex(table_data, f"./results/rules/{params_str}", f"rules_{crossover_type}", f"rules_{params_str}_{crossover_type}", caption)
        write_rules_tex(table_data, prefix_tex_tables, f"rules_{params_str}_{crossover_type}", f"rules_{params_str}_{crossover_type}", caption)
        text = f"\\input{"{"}rules_{params_str}_{crossover_type}_tex{"}"}"
        if macd_params == (12, 26, 9) and crossover_type == 'MACDCROSS':
            write_to_text(prefix_text, "insert_text.txt", text, mode='w')
        else:
            write_to_text(prefix_text, "insert_text.txt", text, mode='a')

In [7]:
table_data

Rule 1 MACD  Rule 1 SMACD  Rule 1 CMACD  \
first    second     third                                               
1_day    one_pos    System 1           13           173           198   
                    System 2           16           117           251   
                    System 3           18           121           245   
         simulation System 1           13           179           192   
                    System 2           16           122           246   
                    System 3           17           120           247   
1_hour   one_pos    System 1           65           657          3054   
                    System 2           91           524          3161   
                    System 3           47           320          3409   
         simulation System 1           60           647          3069   
                    System 2           91           522          3163   
                    System 3           50           313          3413   
5_minute one_pos    System 1           28           157          7655   
                    System 2           34           285          7521   
                    System 3           43           172          7625   
         simulation System 1           27           171          7642   
                    System 2           33           285          7522   
                    System 3           47           177          7616   

                              Rule 2 MACD  Rule 2 SMACD  Rule 2 CMACD  \
first    second     third                                               
1_day    one_pos    System 1            5           261           118   
                    System 2           10           228           146   
                    System 3            8           188           188   
         simulation System 1            5           268           111   
                    System 2           12           227           145   
                    System 3           11           183           190   
1_hour   one_pos    System 1           29          1468          2279   
                    System 2           45          1520          2211   
                    System 3           26           877          2873   
         simulation System 1           28          1476          2272   
                    System 2           46          1519          2211   
                    System 3           29           877          2870   
5_minute one_pos    System 1           17           947          6876   
                    System 2           22          1385          6433   
                    System 3           35           626          7179   
         simulation System 1           17           957          6866   
                    System 2           21          1382          6437   
                    System 3           39           635          7166   

                              Rule 3 MACD  Rule 3 SMACD  Rule 3 CMACD  \
first    second     third                                               
1_day    one_pos    System 1            1           141           242   
                    System 2            2            82           300   
                    System 3            2            78           304   
         simulation System 1            1           146           237   
                    System 2            2            81           301   
                    System 3            3            77           304   
1_hour   one_pos    System 1            4           505          3267   
                    System 2           11           461          3304   
                    System 3            2           242          3532   
         simulation System 1            4           507          3265   
                    System 2           12           460          3304   
                    System 3            2           242          3532   
5_minute one_pos    System 1            6           143          7691   
             